# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring

One row per `(client_hash_id, content_hash_id)` for the month, ranked so an editor knows which content pages to review first for refresh or optimization. The score orders the review queue; it never triggers an automatic content change.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Unit of analysis + time window

**One row = one content item for one client, aggregated across March 2026, ready to be ranked for human refresh review.**

The daily warehouse grain is `report_date × client_hash_id × content_hash_id`. I aggregate it up to one row per `client_hash_id × content_hash_id` for the month.

- **Tables:** `fact_content_daily_performance` (daily GSC measurements) + `dim_content` (creation date).
- **Feature window:** March 1–31, 2026.
- **Outcome window:** April 1–30, 2026 (used only as the ranking target).
- **Sealed:** June 2026 is not read anywhere.
- **Ranked output:** pages ordered by risk of a meaningful next-month visibility decline; top-K goes to an editor.

In [2]:
from pathlib import Path
import os

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import get_token
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
repo_root = Path.cwd()
#repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
extension_dir = repo_root / "work" / "outputs" / ".duckdb_extensions"
extension_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN") or get_token()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert hf_token, "Store a Hugging Face read token as the HF_TOKEN secret; never paste it into a cell."

con = duckdb.connect()
con.execute(f"SET extension_directory='{extension_dir.as_posix()}'")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [hf_token])

ROOT = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{ROOT}/dim_content.parquet')"

print("Authenticated without displaying the token.")
print("Development month: March 2026 | outcome month: April 2026 | June remains sealed")

Authenticated without displaying the token.
Development month: March 2026 | outcome month: April 2026 | June remains sealed


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Features** (knowable March 31) | `impressions`, `ctr`, `avg_position`, `active_days`, `content_age_days` | All observed before the review decision. None encode the outcome. |
| **Label / proxy** | `future_decline = 1` when April impressions < 80% of March impressions (≥ 20 measured GSC days in both months) | This is the ranking target, derived from performance. It can never be an input. |
| **Context** (not modeled) | `client_hash_id`, `content_hash_id`, `report_date`, `content_created_date` | Join keys and split groups. Kept out of the feature matrix. |
| **Excluded (with why)** | All April metrics; product decision fields (`priority_score`, `health_score`, `action_type`); raw IDs as inputs | April is future information at the March decision moment. Product decisions are human labels, not observed outcomes. Raw IDs leak identity. |

In [3]:
# Confirm the columns I'm claiming exist on the March fact partition.
con.sql(f"DESCRIBE SELECT * FROM {FACT_MARCH} LIMIT 1").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries — exactly three

Three queries on the **March 2026 mid-panel partition** (not the sealed `_sample` month). Each proves one contract claim.

1. **Grain** — is `report_date × client_hash_id × content_hash_id` unique?
2. **Slice size and span** — row count, date range, distinct clients and content items.
3. **Availability** — using `IS TRUE`, how many rows survive? This is the honest denominator.

In [4]:
# Query 1 — grain check
grain_query = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_count
FROM {FACT_MARCH}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_violations = con.sql(grain_query).df()
display(grain_violations)
print(f"Duplicate grain groups returned: {len(grain_violations)}")
assert grain_violations.empty, "Grain is not unique — month-level aggregation would double-count."

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


Duplicate grain groups returned: 0


In [5]:
# Query 2 — slice size and date span
slice_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {FACT_MARCH}
"""
slice_facts = con.sql(slice_query).df()
display(slice_facts)

,row_count,min_date,max_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [6]:
# Query 3 — availability with IS TRUE
availability_query = f"""
WITH all_rows AS (
    SELECT COUNT(*) AS total_rows FROM {FACT_MARCH}
), available AS (
    SELECT * FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
)
SELECT
    all_rows.total_rows,
    COUNT(*) AS rows_surviving_is_true,
    ROUND(100.0 * COUNT(*) / all_rows.total_rows, 2) AS percent_surviving
FROM available
CROSS JOIN all_rows
GROUP BY all_rows.total_rows
"""
availability_facts = con.sql(availability_query).df()
display(availability_facts)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_surviving_is_true,percent_surviving
0,9841378,3611061,36.69


## Five-feature frame

I keep **exactly five** predictive features. Each is knowable on **March 31, 2026**, the review decision moment.

1. **`impressions`** — knowable at the decision moment because it sums *measured* March GSC exposure over rows that passed `gsc_data_available IS TRUE`.
2. **`ctr`** — knowable at the decision moment because March clicks and impressions have already occurred. Left missing when the denominator is zero; the imputer handles it instead of writing 0.
3. **`avg_position`** — knowable at the decision moment because it is the March impression-weighted position over valid measured rows only.
4. **`active_days`** — knowable at the decision moment because it counts March days with positive measured impressions; it says how *continuous* presence was, not how large.
5. **`content_age_days`** — knowable at the decision moment because content creation date is metadata already known by March 31.

Hashes are context, not features. `future_decline` is the April target, not a feature.

In [7]:
feature_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(gsc_sum_position) FILTER (
            WHERE gsc_impressions > 0 AND gsc_sum_position > 0
        ) / NULLIF(SUM(gsc_impressions) FILTER (
            WHERE gsc_impressions > 0 AND gsc_sum_position > 0
        ), 0) AS avg_position,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days,
        COUNT(*) AS available_days
    FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
), april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS outcome_impressions,
        COUNT(*) AS outcome_available_days
    FROM {FACT_APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
), content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MIN(content_created_date) AS content_created_date
    FROM {DIM_CONTENT}
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions,
    m.ctr,
    m.avg_position,
    m.active_days,
    GREATEST(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31'), 0) AS content_age_days,
    CASE WHEN a.outcome_impressions < 0.80 * m.impressions THEN 1 ELSE 0 END AS future_decline
FROM march AS m
JOIN april AS a USING (client_hash_id, content_hash_id)
LEFT JOIN content AS c USING (client_hash_id, content_hash_id)
WHERE m.impressions >= 100
  AND m.available_days >= 20
  AND a.outcome_available_days >= 20
ORDER BY m.client_hash_id, m.content_hash_id
"""

examples = con.sql(feature_query).df()
feature_names = ["impressions", "ctr", "avg_position", "active_days", "content_age_days"]
feature_frame = examples[["client_hash_id", "content_hash_id", *feature_names, "future_decline"]].copy()

print(f"Eligible content rows: {len(feature_frame):,}")
print(f"Observed decline rate: {feature_frame['future_decline'].mean():.3f}")
print(f"Predictive feature count: {len(feature_names)}")
assert len(feature_names) == 5
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible content rows: 88,941
Observed decline rate: 0.513
Predictive feature count: 5


,client_hash_id,content_hash_id,impressions,ctr,avg_position,active_days,content_age_days,future_decline
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331.0,0.006042,14.377644,31,175,0
1,client_0797ff3a1fc9a6a5,content_1207efddce873942,461.0,0.000000,14.488069,31,175,0
2,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,232.0,0.000000,11.961207,29,175,0
3,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,774.0,0.006460,12.746770,31,175,0
4,client_0797ff3a1fc9a6a5,content_7beb639d1052e49e,311.0,0.000000,8.540193,30,175,0


### The deliberate leakage trap

For the lesson, I add one forbidden column — an exact copy of the April label — then compare it with the honest five-feature model on the same client-grouped holdout. In a ranking setting the failure is obvious: the leaked model puts every genuinely declining page at the top of the queue with AUC ≈ 1.000, which no real refresh score can do, because at the March decision moment we cannot know April. I then delete that column and keep the honest number.

In [8]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feature_frame, groups=feature_frame["client_hash_id"]))

y = feature_frame["future_decline"]

def quick_auc(columns):
    model = Pipeline([
        ("prep", ColumnTransformer([
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]), columns)
        ])),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    model.fit(feature_frame.iloc[train_idx][columns], y.iloc[train_idx])
    probabilities = model.predict_proba(feature_frame.iloc[test_idx][columns])[:, 1]
    return model, roc_auc_score(y.iloc[test_idx], probabilities), probabilities

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return np.asarray(y_true)[order[:k]].mean()

honest_model, honest_auc, honest_scores = quick_auc(feature_names)
K = max(1, int(0.10 * len(test_idx)))
honest_p_at_k = precision_at_k(y.iloc[test_idx].values, honest_scores, K)

# --- THE TRAP: add one label-derived column on purpose ---
leaked_frame = feature_frame.copy()
leaked_frame["future_decline_copy"] = leaked_frame["future_decline"]
feature_frame = leaked_frame
leaked_model, leaked_auc, leaked_scores = quick_auc([*feature_names, "future_decline_copy"])
leaked_p_at_k = precision_at_k(y.iloc[test_idx].values, leaked_scores, K)

# --- DELETE IT and keep the honest numbers ---
feature_frame = feature_frame.drop(columns="future_decline_copy")
assert "future_decline_copy" not in feature_frame.columns

score_comparison = pd.DataFrame({
    "version": ["honest five features (kept)", "label copy included (rejected)"],
    "grouped_holdout_roc_auc": [honest_auc, leaked_auc],
    f"precision@{K}": [honest_p_at_k, leaked_p_at_k],
})
display(score_comparison.style.format({
    "grouped_holdout_roc_auc": "{:.3f}",
    f"precision@{K}": "{:.3f}",
}))
print(f"Honest AUC: {honest_auc:.3f}  |  honest precision@{K}: {honest_p_at_k:.3f}")
print(f"Leaked AUC: {leaked_auc:.3f}  |  leaked precision@{K}: {leaked_p_at_k:.3f}")
print("Final predictive columns:", feature_names)
print("Leak column present after cleanup:", "future_decline_copy" in feature_frame.columns)

,version,grouped_holdout_roc_auc,precision@2799
0,honest five features (kept),0.664,0.797
1,label copy included (rejected),1.000,1.000


Honest AUC: 0.664  |  honest precision@2799: 0.797
Leaked AUC: 1.000  |  leaked precision@2799: 1.000
Final predictive columns: ['impressions', 'ctr', 'avg_position', 'active_days', 'content_age_days']
Leak column present after cleanup: False


In [9]:
# Ranked human-review queue from the honest model.
# IDs and scores only — no client names, domains, URLs, or raw queries.
test_scores = pd.DataFrame({
    "client_hash_id": feature_frame.iloc[test_idx]["client_hash_id"].values,
    "content_hash_id": feature_frame.iloc[test_idx]["content_hash_id"].values,
    "refresh_score": honest_scores,
    "observed_decline": y.iloc[test_idx].values,
}).sort_values("refresh_score", ascending=False).reset_index(drop=True)

top_queue = test_scores.head(K).copy()
top_queue.insert(0, "rank", range(1, len(top_queue) + 1))
top_queue["recommended_action"] = "review first (refresh or optimize)"

print(f"Ranked review queue size: {len(top_queue):,} of {len(test_scores):,} held-out pages")
print(f"Observed decline rate in top-{K}: {top_queue['observed_decline'].mean():.3f}")
print(f"Observed decline rate in holdout:  {test_scores['observed_decline'].mean():.3f}")
display(top_queue.head(10))

Ranked review queue size: 2,799 of 27,997 held-out pages
Observed decline rate in top-2799: 0.797
Observed decline rate in holdout:  0.667


,rank,client_hash_id,content_hash_id,refresh_score,observed_decline,recommended_action
0,1,client_1a730cb2640a1abf,content_f0ad491eddce6124,0.579921,1,review first (refresh or optimize)
1,2,client_0fa64a184f18a4a0,content_56467aab5997e039,0.579577,1,review first (refresh or optimize)
2,3,client_a80fca3f171ed1de,content_efc50e307a33fee6,0.579419,1,review first (refresh or optimize)
3,4,client_a80fca3f171ed1de,content_c2859e48d6a6750e,0.579406,1,review first (refresh or optimize)
4,5,client_a80fca3f171ed1de,content_d10e63d9ebf90585,0.579398,1,review first (refresh or optimize)
5,6,client_a80fca3f171ed1de,content_e7cfc314417a76e6,0.579323,1,review first (refresh or optimize)
6,7,client_a80fca3f171ed1de,content_f47fb42c8f5bae95,0.579221,1,review first (refresh or optimize)
7,8,client_a80fca3f171ed1de,content_157d381c99e8f2b8,0.579002,1,review first (refresh or optimize)
8,9,client_a80fca3f171ed1de,content_70490f307fc0dde0,0.578984,1,review first (refresh or optimize)
9,10,client_a80fca3f171ed1de,content_9983f7408191c78e,0.578938,0,review first (refresh or optimize)


## 4. Data limits

**One named limitation: coverage-selection bias.** Clients begin GSC tracking at different times, and only rows with `gsc_data_available IS TRUE` survive into the feature frame (Query 3). The eligible March→April slice therefore represents pages with sufficient *measured* coverage, not every page or every client in the warehouse. The ranked queue inherits that selection: a page with sparse or late measurement history is systematically absent from the queue even if it is a real refresh candidate.

Secondary notes:
- **Proxy, not truth.** `future_decline` measures observed search behaviour, not whether a refresh would have helped. The label records an association and cannot prove causation.
- **Month-boundary effects.** March and April differ in days and seasonality; the 80% threshold is a directional rule, not a calibrated target.

In [10]:
# Echo the limitation numerically so the reader sees the size of the selection effect.
print(f"March rows:      {int(slice_facts['row_count'][0]):>12,}")
print(f"Rows IS TRUE:    {int(availability_facts['rows_surviving_is_true'][0]):>12,}")
print(f"Surviving share: {float(availability_facts['percent_surviving'][0]):>11.2f}%")
print(f"Eligible pages:  {len(feature_frame):>12,}")

March rows:         9,841,378
Rows IS TRUE:       3,611,061
Surviving share:       36.69%
Eligible pages:        88,941


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.